In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, Optional

In [ ]:
NOTEBOOK_ROOT = Path.cwd().resolve()

FIXTURE_DIR = NOTEBOOK_ROOT / "fixtures" 
OUTPUT_DIR = NOTEBOOK_ROOT / "rca_runs_case_002"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Python path setup
# ---------------------------------------------------------------------
# Update these if your notebook is in a different location.
PROJECT_SRC = NOTEBOOK_ROOT.parent.parent / "src"
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
from dackar.RCA.orchestrators.rca_reasoning_orchestrator import build_dev_orchestrator
from dackar.RCA.kg.py2neo_workflow import Py2Neo

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", None)

VALIDATOR_MODE = "compat"
STOP_ON_VALIDATION_ERROR = False

# Robust schema path from installed/imported package location
import dackar.RCA.orchestrators.rca_reasoning_orchestrator as orch_mod
SCHEMA_DIR = Path(orch_mod.__file__).resolve().parents[1] / "schemas"

print("Fixture dir:", FIXTURE_DIR)
print("Schema dir :", SCHEMA_DIR)
print("Schemas    :", sorted(p.name for p in SCHEMA_DIR.glob("*.json")))

## Utilities

In [ ]:
def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def maybe_load_json(path: Path) -> Optional[Dict[str, Any]]:
    return load_json(path) if path.exists() else None

def safe_get(d: Optional[Dict[str, Any]], *keys: str, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return default if cur is None else cur

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required fixture file: {path}")

def print_block(title: str, obj: Any, max_chars: int = 5000) -> None:
    print(f"\n--- {title} ---")
    text = json.dumps(obj, indent=2, default=str)
    print(text[:max_chars])

## Load fixture bundle

In [ ]:
required_files = [
    "event.json",
    "telemetry_summary.json",
    "kg_context.json",
]

for name in required_files:
    require_file(FIXTURE_DIR / name)

event = load_json(FIXTURE_DIR / "event.json")
telemetry_summary = load_json(FIXTURE_DIR / "telemetry_summary.json")
kg_context = load_json(FIXTURE_DIR / "kg_context.json")

# Optional prebuilt artifacts
tskr_patterns = maybe_load_json(FIXTURE_DIR / "tskr_patterns.json")
causality_candidates = maybe_load_json(FIXTURE_DIR / "causality_candidates.json")
evidence_bundle = maybe_load_json(FIXTURE_DIR / "evidence_bundle.json")
operational_context = maybe_load_json(FIXTURE_DIR / "operational_context.json")
pm_compliance = maybe_load_json(FIXTURE_DIR / "pm_compliance.json")

## Checks

In [ ]:
assert event["event_id"] == telemetry_summary["event_id"], "event_id mismatch"
assert event["asset_id"] == telemetry_summary["asset_id"], "asset_id mismatch"
assert kg_context["event_id"] == event["event_id"], "kg_context.event_id mismatch"
assert kg_context["asset_id"] == event["asset_id"], "kg_context.asset_id mismatch"

if tskr_patterns is not None:
    assert tskr_patterns["event_id"] == event["event_id"], "tskr_patterns.event_id mismatch"
    assert tskr_patterns["asset_id"] == event["asset_id"], "tskr_patterns.asset_id mismatch"

if causality_candidates is not None:
    assert causality_candidates["event_id"] == event["event_id"], "causality_candidates.event_id mismatch"

if evidence_bundle is not None:
    assert evidence_bundle["retrieval_scope"]["event_id"] == event["event_id"], "evidence_bundle.retrieval_scope.event_id mismatch"
    assert evidence_bundle["retrieval_scope"]["asset_id"] == event["asset_id"], "evidence_bundle.retrieval_scope.asset_id mismatch"

print("Fixture sanity checks passed.")


## Build orchestrator

In [ ]:
client = Py2Neo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

orchestrator = build_dev_orchestrator(
    output_dir=OUTPUT_DIR,
    client=client,
    database=NEO4J_DATABASE,
    schema_dir=SCHEMA_DIR,
    validator_mode=VALIDATOR_MODE,
    stop_on_validation_error=STOP_ON_VALIDATION_ERROR,
)

print("Orchestrator built.")
print("Validator schemas:", sorted(getattr(orchestrator.validator, "schemas", {}).keys()))

## Run orchestrator

In [ ]:
try:
    result = orchestrator.run(
        event=event,
        telemetry_summary=telemetry_summary,
        operational_context=operational_context,
        pm_compliance=pm_compliance,
        kg_context=kg_context,
        tskr_patterns=tskr_patterns,
        causality_candidates=causality_candidates,
        evidence_bundle=evidence_bundle,
    )
finally:
    client.close()

print("Run completed.")

## Inspect outputs

In [ ]:
for key in [
    "input_validation",
    "output_validation",
    "run_manifest",
    "kg_context",
    "tskr_patterns",
    "causality_candidates",
    "evidence_bundle",
    "ishikawa_matrix",
    "rca_card",
]:
    if key in result:
        print_block(key, result[key], max_chars=6000)

summary = {
    "run_id": safe_get(result, "run_context", "run_id"),
    "event_id": safe_get(result, "rca_card", "event_id", default=event["event_id"]),
    "primary_hypothesis": safe_get(result, "rca_card", "primary_hypothesis", default={}),
    "alternative_count": len(safe_get(result, "rca_card", "alternatives", default=[]) or []),
    "n_candidates": len(safe_get(result, "causality_candidates", "candidates", default=[]) or []),
    "n_evidence": len(safe_get(result, "evidence_bundle", "results", default=[]) or []),
    "n_ishikawa_categories": len(safe_get(result, "ishikawa_matrix", "categories", default=[]) or []),
    "input_ok": safe_get(result, "input_validation", "ok"),
    "output_ok": safe_get(result, "output_validation", "ok"),
    "writeback_ready": safe_get(result, "run_manifest", "review_hooks", "writeback_ready"),
}

print_block("summary", summary, max_chars=3000)

## Candidate ranking

In [ ]:
cands = safe_get(result, "causality_candidates", "candidates", default=[]) or []
print("\n=== Candidate ranking ===")
for i, c in enumerate(cands, start=1):
    print(
        f"{i}. {c.get('candidate_id')} | "
        f"{c.get('cause_label')} | "
        f"score={c.get('composite_score')} | "
        f"temporal={safe_get(c, 'temporal_evidence', 'relation')} | "
        f"signals={safe_get(c, 'telemetry_evidence', 'matching_signal_ids', default=[])}"
    )

# Ishikawa summary
cats = safe_get(result, "ishikawa_matrix", "categories", default=[]) or []
print("\n=== Ishikawa categories ===")
for cat in cats:
    category = cat.get("category")
    rows = cat.get("rows", []) or []
    print(f"{category}: {len(rows)} rows")
    for row in rows[:3]:
        print(
            "   -",
            row.get("label"),
            "| strength=",
            row.get("strength"),
            "| source=",
            row.get("source_artifact"),
        )

## Full result bundle

In [ ]:
out_path = OUTPUT_DIR / f"{summary['run_id']}_full_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, default=str)

print("Saved full result to:", out_path)